In [0]:
# ============================================================
# Create Delta Tables from S3 for Databricks Dashboard
# Run this cell once — tables persist permanently
# ============================================================

import boto3
import pandas as pd
import io
import os
from dotenv import load_dotenv
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

# Load credentials
load_dotenv("/Workspace/Users/hsp498.ca@gmail.com/.env")
# AWS Credentials — used for Excel file reads only
ACCESS_KEY = os.getenv("ACCESS_KEY")
SECRET_KEY  = os.getenv("SECRET_KEY")
BUCKET      = os.getenv("BUCKET")
REGION      = os.getenv("REGION")

s3 = boto3.client(
    's3',
    aws_access_key_id     = ACCESS_KEY,
    aws_secret_access_key = SECRET_KEY,
    region_name           = REGION
)

print("Creating Delta tables...\n")

# Create database
spark.sql("CREATE DATABASE IF NOT EXISTS ecom_project")

def save_delta(s3_key, table_name):
    """Load parquet from S3 and save as Delta table"""
    print(f"Processing {table_name}...")
    obj = s3.get_object(Bucket=BUCKET, Key=s3_key)
    pdf = pd.read_parquet(io.BytesIO(obj['Body'].read()))

    # Clean column names
    pdf.columns = [c.replace(' ', '_').replace('-', '_') for c in pdf.columns]

    # Fix timestamps
    for col in pdf.columns:
        if pd.api.types.is_datetime64_any_dtype(pdf[col]):
            pdf[col] = pdf[col].astype('datetime64[us]')

    # Save as Delta
    sdf = spark.createDataFrame(pdf)
    for field in sdf.schema.fields:
        if 'timestamp' in str(field.dataType).lower():
            sdf = sdf.withColumn(field.name, F.col(field.name).cast(TimestampType()))

    sdf.write.format("delta").mode("overwrite").saveAsTable(f"ecom_project.{table_name}")
    count = spark.sql(f"SELECT COUNT(*) AS cnt FROM ecom_project.{table_name}").collect()[0][0]
    print(f"  Done — {count:,} rows saved\n")

# Create all 6 tables
save_delta("outputs/master_customer_dataset.parquet",       "master_customers")
save_delta("outputs/rfm_segments.parquet",                  "rfm_segments")
save_delta("outputs/customer_survival_data.parquet",        "survival_data")
save_delta("outputs/at_risk_customers.parquet",             "at_risk_customers")
save_delta("outputs/product_sentiment.parquet",             "product_sentiment")
save_delta("outputs/all_customer_recommendations.parquet",  "recommendations")

# Verify all tables
print("=== All Delta Tables Created ===")
spark.sql("SHOW TABLES IN ecom_project").show(truncate=False)

# Test query
print("Test query:")
spark.sql("""
    SELECT RFM_Segment,
           COUNT(*)                   AS Customers,
           ROUND(SUM(Monetary), 0)    AS Revenue,
           ROUND(AVG(churned)*100, 1) AS Churn_Pct
    FROM ecom_project.master_customers
    GROUP BY RFM_Segment
    ORDER BY Revenue DESC
""").show()

print("All Delta tables ready — go to Dashboards tab now!")

Creating Delta tables...

Processing master_customers...
  Done — 5,878 rows saved

Processing rfm_segments...
  Done — 5,878 rows saved

Processing survival_data...
  Done — 5,878 rows saved

Processing at_risk_customers...
  Done — 497 rows saved

Processing product_sentiment...
  Done — 20,392 rows saved

Processing recommendations...
  Done — 29,390 rows saved

=== All Delta Tables Created ===
+------------+-----------------+-----------+
|database    |tableName        |isTemporary|
+------------+-----------------+-----------+
|ecom_project|at_risk_customers|false      |
|ecom_project|master_customers |false      |
|ecom_project|product_sentiment|false      |
|ecom_project|recommendations  |false      |
|ecom_project|rfm_segments     |false      |
|ecom_project|survival_data    |false      |
+------------+-----------------+-----------+

Test query:
+---------------+---------+-----------+---------+
|    RFM_Segment|Customers|    Revenue|Churn_Pct|
+---------------+---------+---------

In [0]:
# Run this and paste the output here
print("=== master_customers ===")
spark.sql("SELECT RFM_Segment, COUNT(*) AS Customers, ROUND(SUM(Monetary),2) AS Revenue, ROUND(AVG(churned)*100,1) AS Churn_Pct FROM ecom_project.master_customers GROUP BY RFM_Segment ORDER BY Revenue DESC").show()

print("=== at_risk_customers ===")
spark.sql("SELECT COUNT(*) AS At_Risk FROM ecom_project.at_risk_customers").show()

print("=== recommendations ===")
spark.sql("SELECT Description, ROUND(SUM(Score),0) AS Score FROM ecom_project.recommendations GROUP BY Description ORDER BY Score DESC LIMIT 10").show()

print("=== product_sentiment ===")
spark.sql("SELECT CASE WHEN avg_sentiment >= 0.5 THEN 'Very Positive' WHEN avg_sentiment >= 0.05 THEN 'Positive' WHEN avg_sentiment >= -0.05 THEN 'Neutral' ELSE 'Negative' END AS Band, COUNT(*) AS Products FROM ecom_project.product_sentiment GROUP BY Band ORDER BY Band").show()

=== master_customers ===
+---------------+---------+-------------+---------+
|    RFM_Segment|Customers|      Revenue|Churn_Pct|
+---------------+---------+-------------+---------+
|      Champions|     1270|1.275266867E7|      4.7|
|Loyal Customers|     1316|   3113387.02|     32.1|
|        At Risk|     1406|   1289163.66|     58.6|
|           Lost|     1886|    588209.83|     89.0|
+---------------+---------+-------------+---------+

=== at_risk_customers ===
+-------+
|At_Risk|
+-------+
|    497|
+-------+

=== recommendations ===
+--------------------+--------+
|         Description|   Score|
+--------------------+--------+
|RED  HARMONICA IN...|243594.0|
|BAG 250g SWIRLY M...|216313.0|
|WHITE HANGING HEA...|186811.0|
|PACK OF 72 SKULL ...|174785.0|
|PACK OF 72 RETRO ...|154921.0|
|PACK OF 12 PINK P...|154100.0|
|72 SWEETHEART FAI...|145724.0|
|PACK OF 60 PINK P...|144571.0|
|60 TEATIME FAIRY ...|141207.0|
|PACK OF 12 WOODLA...|131243.0|
+--------------------+--------+

=== prod